# L01 Reflective Journal – VGG16 Pre-trained on ImageNet
**Name:** Yoana Cook
**Professor:** Patricia McManus  
**Course:** ITAI 2376 – Deep Learning  
**Date:** February 14, 2026  
**Lab:** L01 – Introduction to Deep Learning with VGG16


---
## 1. Introduction

### Task
Read the introduction provided in the markdown cell and understand the goals of the lab.

### Reflection
This lab provides a hands-on introduction to deep learning by walking through a complete workflow using VGG16, a convolutional neural network pre-trained on the ImageNet dataset. The goal is not to write code from scratch but to observe and understand each stage of a deep learning pipeline: loading a model, preparing data, training through transfer learning, making predictions, and evaluating results.

Deep learning models like VGG16 learn hierarchical feature representations from images, early layers capture edges and textures, while deeper layers capture more complex patterns like shapes and object parts. By using a model already trained on over 1 million images across 1,000 categories, we can apply transfer learning: reusing knowledge learned on one task and adapting it to a new, related task.

In this lab, the new task is **binary classification of scorpion peppers vs. non-scorpion peppers**, which is a much narrower problem than ImageNet's 1,000-class challenge. Transfer learning makes this feasible even with a relatively small dataset.


---
## 2. Overview of the Pre-built Model

### Task
Load the VGG16 model and view its architecture.

### Observations
The VGG16 model was loaded with pre-trained ImageNet weights:
```python
vggmodel = VGG16(weights="imagenet", include_top='False', input_shape=(224,224,3))
```

The architecture consists of 5 convolutional blocks:
- **Block 1:** 2 Conv2D layers (64 filters) + MaxPooling
- **Block 2:** 2 Conv2D layers (128 filters) + MaxPooling
- **Block 3:** 3 Conv2D layers (256 filters) + MaxPooling
- **Block 4:** 3 Conv2D layers (512 filters) + MaxPooling
- **Block 5:** 3 Conv2D layers (512 filters) + MaxPooling

For transfer learning, the top 4 layers of VGG16 were removed, keeping blocks 1–5 (through `block5_pool`). These layers were then **frozen** (set to non-trainable) so their learned weights are preserved. A new classifier was added on top:
- Flatten layer
- Dense(1024, relu)
- Dense(512, relu)
- Dense(1, sigmoid) — single output for binary classification

### Reflection
The progressive increase in filter depth (64 → 128 → 256 → 512) while spatial dimensions shrink through pooling is a core CNN design pattern. The network is essentially trading spatial resolution for richer, more abstract feature representations as data moves deeper through the layers.

VGG16's design is notably uniform — it uses only 3×3 convolutions throughout, making it straightforward to understand compared to more complex architectures like Inception or ResNet. Freezing the convolutional base is key: these layers already know how to extract useful visual features from their ImageNet training, and we only need to teach the new top layers how to map those features to our specific two-class problem (scorpion pepper vs. not scorpion pepper).


---
## 3. Data Loading and Preprocessing

### Task
Observe data loading and preprocessing and understand why each step matters.

### Observations
The dataset was split into three sets:
- **Training:** 2,738 images (2 classes)
- **Validation:** 686 images (2 classes)
- **Test:** 856 images (2 classes)

The class labels were: `{'not_scorpion': 0, 'scorpion_pepper': 1}`

**Training data** used extensive augmentation:
```python
trdata = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
```

**Validation and test data** used only rescaling — no augmentation:
```python
valdata = ImageDataGenerator(rescale=1./255)
tsdata = ImageDataGenerator(rescale=1./255)
```

All images were resized to 224×224 pixels (VGG16's required input size) and loaded in batches of 20.

### Reflection
Data preprocessing is one of the most critical steps in any deep learning project. Several things stood out to me:

**Rescaling (1./255):** This normalizes pixel values from [0, 255] to [0, 1]. Neural networks train more effectively when input values are small and on a similar scale, because it helps gradient descent converge faster and more stably.

**Augmentation only on training data:** The training augmentations (rotation, shifting, shearing, zooming, flipping) artificially expand the effective size of the training set by showing the model different variations of each image. This reduces overfitting. However, validation and test data are deliberately left unaugmented because we want to evaluate the model on realistic, unmodified images — the same kind it would see in production.

**fill_mode='nearest':** When transformations like rotation create empty pixel areas at image edges, this setting fills them by repeating the nearest pixel value rather than inserting black borders. This is a thoughtful choice because black borders could teach the model that "edges with black = transformed image," which is not a useful feature to learn.

**Binary class mode:** Since this is a two-class problem (scorpion pepper vs. not), `class_mode='binary'` produces single scalar labels (0 or 1) rather than one-hot vectors, which pairs correctly with the sigmoid output and binary cross-entropy loss used later.


---
## 4. Making Predictions

### Task
Observe the model training and making predictions.

### Observations
The model was compiled and trained with:
```python
mymodel.compile(optimizers.Adam(lr=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
history = mymodel.fit_generator(generator=traindata, steps_per_epoch=137,
                                validation_data=validatedata, validation_steps=35,
                                epochs=15, verbose=2)
```

**Training progress (selected epochs):**

| Epoch | Train Loss | Train Acc | Val Loss | Val Acc |
|-------|-----------|-----------|----------|---------|
| 1     | 0.3932    | 82.18%    | 0.1743   | 93.44%  |
| 3     | 0.1631    | 93.61%    | 0.0854   | 96.94%  |
| 5     | 0.1084    | 95.54%    | 0.0854   | ~96%    |
| 15    | (final)   | ~96%      | (final)  | ~97%    |

The model improved rapidly in the first few epochs and then continued making steady gains. Validation accuracy reached into the high 90s, closely tracking training accuracy.

### Reflection
Several aspects of the training configuration are worth noting:

**Learning rate of 0.0001:** This is intentionally small. Since the convolutional base is frozen and we are only training the small classifier on top, a low learning rate helps those new layers learn stable, well-calibrated weights without wild updates.

**Steps per epoch = 137 with batch size 20:** This means 137 × 20 = 2,740 images per epoch, which aligns with the 2,738 training images found. So each epoch processes essentially the entire training set once.

**Rapid initial improvement:** The jump from 82% to 93% accuracy in just the first few epochs shows the power of transfer learning. The frozen VGG16 layers are already providing excellent feature representations — the classifier just needs to learn how to separate two classes based on those features.

**Validation tracking training:** The fact that validation accuracy closely follows training accuracy (and validation loss stays relatively low) suggests the model is generalizing well and not significantly overfitting, which is a good sign given the augmentation and frozen-layer strategy.


---
## 5. Understanding Predictions

### Task
Observe how the model performs on test data and understand what affects predictions.

### Observations
The model was evaluated on the 856-image test set, producing a classification report:

```
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       428
           1       0.93      1.00      0.96       428

    accuracy                           0.96       856
   macro avg       0.96      0.96      0.96       856
weighted avg       0.96      0.96      0.96       856
```

**Key metrics:**
- **Overall accuracy: 96%** across 856 test images
- **Class 0 (not_scorpion):** Perfect precision (1.00) but 92% recall — when it says "not scorpion," it's always right, but it misses 8% of actual non-scorpion images
- **Class 1 (scorpion_pepper):** 93% precision with perfect recall (1.00) — it catches every single scorpion pepper, but 7% of its "scorpion pepper" predictions are actually non-scorpion images

The accuracy and loss plots showed the training and validation curves converging, with the model reaching strong performance by around epoch 5 and continuing to improve slightly through epoch 15.

### Reflection
The classification report reveals an interesting asymmetry in the model's errors:

The model has a **conservative bias toward classifying images as scorpion peppers** — it never misses a real scorpion pepper (100% recall for class 1), but it sometimes incorrectly labels a non-scorpion image as a scorpion pepper (8% of non-scorpion images are misclassified). This pattern suggests the model may have learned that "when in doubt, call it a scorpion pepper."

Understanding the **precision vs. recall trade-off** is important in real-world applications. If this model were used in an agricultural sorting system, the current behavior (catching all scorpion peppers at the cost of some false positives) might be acceptable or even preferred. In other contexts, you might want to adjust the decision threshold from 0.5 to shift this balance.

The **balanced test set** (428 images per class) is also important. With equal class representation, the 96% accuracy is meaningful. If the classes were heavily imbalanced (e.g., 95% non-scorpion), even a model that always guessed "not scorpion" would appear to have 95% accuracy while being completely useless.

Regarding input sensitivity: image modifications like rotation beyond the training augmentation range, added noise, or significant color shifts would likely degrade performance because they push the input outside the distribution the model was trained on. The training augmentation of ±40° rotation, for example, means the model has some rotation tolerance but would struggle with upside-down images.


---
## 6. Conclusion and Discussion

### Discussion Questions

**Q1: What are the main components of a deep learning model?**

A deep learning model consists of an input layer that accepts data in a specific format (224×224×3 RGB images for VGG16), hidden layers that progressively extract and transform features through learnable weights and activation functions, and an output layer that produces the final prediction. In CNNs specifically, the hidden layers include convolutional layers (which apply learned filters to detect features), pooling layers (which reduce spatial dimensions while retaining important information), and fully connected/dense layers (which combine features for classification). Beyond the architecture itself, a complete model requires a loss function (binary cross-entropy in this lab) to quantify prediction error, an optimizer (Adam) to iteratively adjust weights through backpropagation, and properly preprocessed training data.

**Q2: Why is data preprocessing important in deep learning?**

Preprocessing ensures the input data matches what the model expects and helps the model learn effectively. In this lab, rescaling to [0,1] normalized pixel values so that gradient descent operates on a manageable scale. Resizing to 224×224 matched VGG16's fixed input requirements. Data augmentation was critical because with only 2,738 training images, the model could easily memorize the training set (overfit) without seeing enough variation. By applying random rotations, shifts, flips, and zooms, each epoch effectively shows the model different versions of every image, teaching it to focus on the actual visual features of scorpion peppers rather than memorizing specific pixel arrangements. The fact that augmentation was applied only to training data — not validation or test — preserves the integrity of the evaluation.

**Q3: How does transfer learning work, and why is it useful?**

Transfer learning reuses a model trained on a large, general dataset for a different but related task. In this lab, VGG16's convolutional layers learned to extract visual features (edges, textures, shapes, object parts) from 1.2 million ImageNet images. These features are broadly useful for many vision tasks — an edge detector trained on dogs and cars still detects edges in pepper images. By freezing those layers and only training a small new classifier, we get three major benefits: (1) we need far less task-specific data (2,738 images vs. millions), (2) training is much faster (minutes vs. days), and (3) we often achieve better results than training a small model from scratch, because we're building on rich, well-learned representations.

**Q4: What factors affect a model's prediction confidence?**

From the lab, several factors affect how confidently and correctly the model classifies: the similarity of the input to the training data distribution (images that look very different from training examples will produce less reliable predictions), image quality and resolution, transformations like rotation or noise that push images outside the augmented range, visual ambiguity between classes (a non-scorpion pepper that happens to look similar to a scorpion pepper), and the decision threshold (0.5 in this lab, but adjustable). The training itself also matters — a model trained for more epochs or with a better learning rate schedule may produce more calibrated confidence scores.

**Q5: What are the limitations of this approach?**

The main limitations include: this model only handles binary classification between two specific categories and cannot be easily extended without retraining; VGG16 is computationally heavy with ~138 million parameters (more modern architectures like EfficientNet achieve similar or better accuracy with far fewer parameters); the fixed 224×224 input size forces all images to be resized, which can distort images with unusual aspect ratios; the model inherits any biases present in both the ImageNet pre-training data and the scorpion pepper dataset; and as demonstrated, the model is sensitive to input modifications that fall outside its training distribution. The use of the deprecated `fit_generator` (now replaced by `fit`) also suggests this codebase could benefit from updating to current Keras best practices.


---
## 7. Feedback and Self-Assessment

### Lab Feedback
This lab was a well-structured introduction to deep learning. The progression from model architecture → preprocessing → training → evaluation followed the natural pipeline of a real deep learning project, which made it easy to connect each step to the bigger picture. The pre-loaded code cells with existing outputs were helpful for understanding what each block does without getting stuck on environment setup or dependency issues.

The use of a concrete, visual classification task (scorpion peppers) made the abstract concepts tangible. Seeing the model go from 82% accuracy in epoch 1 to 96% on the final test set made the power of transfer learning very real.

One area for improvement: it would be valuable to include a cell that visualizes some of the misclassified images. Seeing which specific images the model got wrong (the ~4% errors) would build intuition about what confuses the model and what the boundaries of its learning are.

### Self-Assessment
After completing this lab, I can confidently:

- **Describe VGG16's architecture** and explain the role of convolutional, pooling, and dense layers
- **Explain why each preprocessing step matters** — rescaling, resizing, augmentation, and why augmentation is applied only to training data
- **Articulate how transfer learning works** — freezing a pre-trained base and training a new classifier head
- **Interpret a classification report** — understanding precision, recall, F1-score, and what the asymmetry between classes means in practice
- **Identify factors that affect model robustness** — input perturbations, distribution shift, and the relationship between training augmentation and test-time tolerance
- **Recognize limitations** of this approach and where modern alternatives (like EfficientNet or ResNet) improve upon VGG16
